### RAG Pipieline - Data Ingestion to VectorDB Pipeline

In [9]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from pathlib import Path

In [43]:
## Read all PDF files from a directory and load them into a list of documents

def process_all_pdfs_in_directory(directory_path):
    documents = []
    pdf_dir=Path(directory_path)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory: {directory_path}")
    for pdf in pdf_files:
        print(f"Processing PDF file: {pdf}")
        try:
            loader = PyMuPDFLoader(str(pdf))
            pdf_documents = loader.load()
            for doc in pdf_documents:
                doc.metadata["source"] = str(pdf)
                doc.metadata["file_type"] = "pdf"
            documents.extend(pdf_documents)
            print(f"Successfully processed {len(pdf_documents)} documents from {pdf}")
        except Exception as e:
            print(f"Error occurred while processing {pdf}: {e}")
    print(f"Total documents loaded from all PDFs: {len(documents)}")
    print(documents)
    return documents
    

In [44]:
pdf_documents=process_all_pdfs_in_directory("../data")
pdf_documents

Found 2 PDF files in the directory: ../data
Processing PDF file: ..\data\pdf\file-example_PDF_500_kB.pdf
Successfully processed 5 documents from ..\data\pdf\file-example_PDF_500_kB.pdf
Processing PDF file: ..\data\pdf\file-sample_150kB.pdf
Successfully processed 4 documents from ..\data\pdf\file-sample_150kB.pdf
Total documents loaded from all PDFs: 9
[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'file_path': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20170816144228+02'00'", 'page': 0, 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent 

[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'file_path': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20170816144228+02'00'", 'page': 0, 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum. Vivamus dapibus sodales ex, vitae malesuada ipsum cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci. Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. I

In [53]:
### TextSplitting getting into chunks of text from the documents

def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    """Split the documents into chunks of text."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(split_docs)}")
    if split_docs:
        print(f"First chunk: {split_docs[0].page_content[:500]}...")  # Print first 500 characters of the first chunk
        print(f"Metadata of first chunk: {split_docs[0].metadata}")
    return split_docs

In [54]:
chunks=split_documents_into_chunks(pdf_documents)
chunks

Total chunks created: 21
First chunk: Lorem ipsum 
Lorem ipsum dolor sit amet, consectetur adipiscing 
elit. Nunc ac faucibus odio. 
Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut
varius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum
condimentum. Vivamus dapibus sodales ex, vitae malesuada ipsum cursus
convallis. Maecenas sed egestas nulla, ac condimentum orci. Mauris diam felis,
vulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula sempe...
Metadata of first chunk: {'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'file_path': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20170816144228+02'00'", 'page': 0, 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'file_path': '..\\data\\pdf\\file-example_PDF_500_kB.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20170816144228+02'00'", 'page': 0, 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum. Vivamus dapibus sodales ex, vitae malesuada ipsum cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci. Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. I

### Embedding and VectorStoreDB

In [55]:
import uuid
import numpy as np      
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any, Tuple
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity  

In [61]:
class EmbeddingManager:
    """Handles documenent embeddings generation using SentenceTransformers"""

    def __init__(self,model_name: str ="all-MiniLM-L6-v2"):
        """Initialises the embedding manager
        Args:
            model_name (str): HuggingFace model name for generating embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded model: {self.model_name} with embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to generate embeddings for.
        Returns:
            np.ndarray: Array of embeddings.
        """
        if not self.model:  
            raise ValueError("Model is not loaded. Please check the model name or loading process.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    ## initilise the embedding manger
    embedding_manager = EmbeddingManager()
    embedding_manager
    

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5408.35it/s]


Successfully loaded model: all-MiniLM-L6-v2 with embedding dimension: 384


## VectorStore

In [ ]:
class VectorStore:
    """Manages a vector store for document embeddings using ChromaDB."""
    def __init__(self,collection_name: str ="pdf_documents",persist_directory: str ="../data/vector_store"):
        """
        Initializes the VectorStore with a ChromaDB client and collection.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persist_directory (str): Directory to persist the vector store.
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromadb client and collection."""
        try:
            # Create Persist ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistant_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")
        except Exception as e:
            print("Error initializing vector store:", e)
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adds documents and their embeddings to the vector store.

        Args:
            documents (List[Any]): List of document objects.
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """
